# PREP_01 - Train /  Test / Validation Split

## TFM - Skin Lesion Classification (ISIC 2024 / SLICE-3D)

Con el objetivo de obtener los datasets que utilizaremos para entrenar nuestros modelos de clasificación en este notebook vamos a trabajar a partir de los datasets resultantes de los scripts preprocess_data.py y final_preprocess_data.py

En concreto trabajaremos con el artefacto ya generado:

* **Final preprocessed metadata** (`final_preprocessed_from_raw_<timestamp>.parquet`) — Es el resultado de EDA 01 y 02 y contiene todos los datos que necesitamos para hacer el split.



## Imports

In [2]:
import pandas as pd
import numpy as np


from sklearn.model_selection import StratifiedGroupKFold


from skin_lesion_ai.utils.data_utils import (
    load_metadata_parquet,
)

from skin_lesion_ai.visualisation.eda_plots import (
    set_eda_style,
)

# Set the EDA style
set_eda_style()

## Cargar Dataset

Cargamos el dataset utilizando la función creada para tal efecto. 

In [3]:
# Load the preprocessed metadata
df_preprocessed = load_metadata_parquet(
    stage="processed",
    filename="final_preprocessed_from_raw",
    timestamp_flag=True,
)

print(f"Final preprocessed metadata: {df_preprocessed.shape}")
df_preprocessed.head()

Final preprocessed metadata: (381280, 17)


,isic_id,patient_id,diagnostic_group,target_biopsy,target_malignant,sex,sex_male,age_approx,anatom_site_general,anatom_site_general_code,anatom_site__anterior_torso,anatom_site__head_neck,anatom_site__lower_extremity,anatom_site__posterior_torso,anatom_site__upper_extremity,clin_size_long_diam_mm,clin_size_long_diam_mm_log1p
0,ISIC_0015670,IP_1235828,benign_non_biopsied,0,<NA>,male,1,60.0,lower extremity,4,0,0,1,0,0,3.04,1.396245
1,ISIC_0015845,IP_8170065,benign_non_biopsied,0,<NA>,male,1,60.0,head/neck,5,0,1,0,0,0,1.10,0.741937
2,ISIC_0015864,IP_6724798,benign_non_biopsied,0,<NA>,male,1,60.0,posterior torso,2,0,0,0,1,0,3.40,1.481605
3,ISIC_0015902,IP_4111386,benign_non_biopsied,0,<NA>,male,1,65.0,anterior torso,1,1,0,0,0,0,3.22,1.439835
4,ISIC_0024200,IP_8313778,benign_non_biopsied,0,<NA>,male,1,55.0,anterior torso,1,1,0,0,0,0,2.73,1.316408


In [4]:
# Comprobamos la información del DataFrame preprocesado
df_preprocessed.info()

<class 'pandas.DataFrame'>
RangeIndex: 381280 entries, 0 to 381279
Data columns (total 17 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   isic_id                       381280 non-null  str    
 1   patient_id                    381280 non-null  str    
 2   diagnostic_group              381280 non-null  string 
 3   target_biopsy                 381280 non-null  int8   
 4   target_malignant              1013 non-null    Int8   
 5   sex                           381280 non-null  str    
 6   sex_male                      381280 non-null  int8   
 7   age_approx                    381280 non-null  float64
 8   anatom_site_general           381280 non-null  str    
 9   anatom_site_general_code      381280 non-null  int8   
 10  anatom_site__anterior_torso   381280 non-null  int8   
 11  anatom_site__head_neck        381280 non-null  int8   
 12  anatom_site__lower_extremity  381280 non-null  int8   


## Using splitter to split df stratified and by groupal id

Para hacer la partición del dataset en los correspondientes train / test / validation vamos a utilizar como splitter la función StratifiedGroupKFold ya que nos va a permitir aplicar a la partición todas las condiciones que nos interesa para operar.

Con StratifiedGroupKFold podemos hacer que la partición se haga agrupando las lesiones por paciente (grouping), que trate de respetar la estructura de 'target_biopsy' (stratification) y haga shuffle. La limitación que tenemos con StratifiedGroupKFold es que no en todos los casos obtendremos particiones del tamaño que deseamos, ya que está diseñada para cross-validation y trabaja a partir del número de splits que pretendemos hacer.


Para más información podemos consultar las fuentes donde hemos aprendido:


Issue discusion on how/why use the function StratifiedGroupKFold to do the split: https://github.com/scikit-learn/scikit-learn/issues/12076

Detailed explanation on split: https://github.com/scikit-learn/scikit-learn/issues/9193

Aplied example: https://stackoverflow.com/questions/56872664/complex-dataset-split-stratifiedgroupshufflesplit


In [5]:
# Split patients into train and test_val sets
# El test_size esta relacionado inversamente con el numero de folds
# Pero el n_folds debe ser entero

random_state = 42
test_size = 0.2
desired = 1.0 / test_size
n_folds = int(np.round(desired))
# De manera alternativa: c=np.ceil(desired), f=np.floor(desired), c if c/desired < desired /f else f

# Creamos objeto splitter
splitter = StratifiedGroupKFold(
    n_splits=n_folds, shuffle=True, random_state=random_state
)


# Split manteniendo stratificacion de 'target_biopsy' y agrupando por 'patient_id'
split = splitter.split(
    X=df_preprocessed,
    y=df_preprocessed["target_biopsy"],
    groups=df_preprocessed["patient_id"],
)
train_inds, test_val_inds = next(split)

train_df = df_preprocessed.iloc[train_inds]
test_val_df = df_preprocessed.iloc[test_val_inds]

In [6]:
# Debemos hacer split de nuevo para separar test y validation

val_size = 0.5
desired_val = 1.0 / val_size
n_folds_val = int(np.round(desired_val))

# Creamos obj splitter para el split que haremos
splitter_val = StratifiedGroupKFold(
    n_splits=n_folds_val, shuffle=True, random_state=random_state
)

# Creamos el obj split manteniendo stratificacion de 'target_biopsy' y agrupando por 'patient_id'
split_val = splitter_val.split(
    X=test_val_df, y=test_val_df["target_biopsy"], groups=test_val_df["patient_id"]
)

# Aplicamos el split 1 iteracion
test_inds, val_inds = next(split_val)

test_df = test_val_df.iloc[test_inds]
val_df = test_val_df.iloc[val_inds]

## Validacion de los splits

El chequeo más inmediato que podemos hacer es comprobar la longitud de los sets.

In [7]:
# Check train and test sets length
print(f"Train set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"Val set: {len(val_df)} samples")

# Percentage of the original dataset
print(f"Train set: {len(train_df) * 100 / len(df_preprocessed):.2f}% samples")
print(f"Test set: {len(test_df) * 100 / len(df_preprocessed):.2f}% samples")
print(f"Val set: {len(val_df) * 100 / len(df_preprocessed):.2f}% samples")

Train set: 305024 samples
Test set: 38125 samples
Val set: 38131 samples
Train set: 80.00% samples
Test set: 10.00% samples
Val set: 10.00% samples


Vamos a comprobar que no haya ningún paciente en ambos splits a la vez.

In [ ]:
# Patient check overlaps

df_overlap1 = pd.merge(train_df, test_df, how="inner", on=["patient_id", "patient_id"])
print(f"Train ∩ Test overlap: {len(df_overlap1)} patients")

df_overlap2 = pd.merge(val_df, test_df, how="inner", on=["patient_id", "patient_id"])
print(f"Validation ∩ Test overlap: {len(df_overlap2)} patients")

df_overlap3 = pd.merge(val_df, train_df, how="inner", on=["patient_id", "patient_id"])
print(f"Validation ∩ Train overlap: {len(df_overlap3)} patients")

if (len(df_overlap1)) + len(df_overlap2) + len(df_overlap3) == 0:
    print("No patient overlap")
else:
    print("Patient overlap detected")

Train ∩ Test overlap: 0 patients
Validation ∩ Test overlap: 0 patients
Validation ∩ Train overlap: 0 patients
✅ No patient overlap


Verificamos que la partición la hemos hecho stratificada y, por lo tanto, mantenemos las proporciones en el split.

In [9]:
# checkear stratificacion de nuestro split

preprocessed_y = df_preprocessed["target_biopsy"].value_counts()
train_y = train_df["target_biopsy"].value_counts()
test_y = test_df["target_biopsy"].value_counts()
val_y = val_df["target_biopsy"].value_counts()

print(
    f"Preprocessed_df target_biopsy samples count:\n {preprocessed_y[0]} with value '0' \n {preprocessed_y[1]} with value '1' \n"
)
print(
    f"As percentatge:\n {100 * preprocessed_y[0] / len(df_preprocessed):.2f}% with value '0' \n {100 * preprocessed_y[1] / len(df_preprocessed):.2f}% with value '1' \n"
)

print("\nAfter our split we have: \n")

print(
    f"Train_df target_biopsy samples count:\n {train_y[0]} with value '0' \n {train_y[1]} with value '1' \n"
)
print(
    f"As percentatge:\n {100 * train_y[0] / len(train_df):.2f}% with value '0' \n {100 * train_y[1] / len(train_df):.2f}% with value '1' \n"
)

print(
    f"Test_df target_biopsy samples count:\n {test_y[0]} with value '0' \n {test_y[1]} with value '1' \n"
)
print(
    f"As percentatge:\n {100 * test_y[0] / len(test_df):.2f}% with value '0' \n {100 * test_y[1] / len(test_df):.2f}% with value '1' \n"
)

print(
    f"Val_df target_biopsy samples count:\n {val_y[0]} with value '0' \n {val_y[1]} with value '1' \n"
)
print(
    f"As percentatge:\n {100 * val_y[0] / len(val_df):.2f}% with value '0' \n {100 * val_y[1] / len(val_df):.2f}% with value '1' \n"
)

Preprocessed_df target_biopsy samples count:
 380267 with value '0' 
 1013 with value '1' 

As percentatge:
 99.73% with value '0' 
 0.27% with value '1' 


After our split we have: 

Train_df target_biopsy samples count:
 304212 with value '0' 
 812 with value '1' 

As percentatge:
 99.73% with value '0' 
 0.27% with value '1' 

Test_df target_biopsy samples count:
 38025 with value '0' 
 100 with value '1' 

As percentatge:
 99.74% with value '0' 
 0.26% with value '1' 

Val_df target_biopsy samples count:
 38030 with value '0' 
 101 with value '1' 

As percentatge:
 99.74% with value '0' 
 0.26% with value '1' 



In [ ]:
# Checkear stratification para cada split en % hasta 1 decimal

if np.round(100 * preprocessed_y[0] / len(df_preprocessed), 1) == np.round(
    100 * train_y[0] / len(train_df), 1
):
    print("Stratification maintained for value '0' in train set")
else:
    print("Stratification not maintained for value '0' in train set")

if np.round(100 * preprocessed_y[1] / len(df_preprocessed), 1) == np.round(
    100 * train_y[1] / len(train_df), 1
):
    print("Stratification maintained for value '1' in train set")
else:
    print("Stratification not maintained for value '1' in train set")

# Check stratification for test set
if np.round(100 * preprocessed_y[0] / len(df_preprocessed), 1) == np.round(
    100 * test_y[0] / len(test_df), 1
):
    print("Stratification maintained for value '0' in test set")
else:
    print("Stratification not maintained for value '0' in test set")

if np.round(100 * preprocessed_y[1] / len(df_preprocessed), 1) == np.round(
    100 * test_y[1] / len(test_df), 1
):
    print("Stratification maintained for value '1' in test set")
else:
    print("Stratification not maintained for value '1' in test set")

# Check stratification for validation set
if np.round(100 * preprocessed_y[0] / len(df_preprocessed), 1) == np.round(
    100 * val_y[0] / len(val_df), 1
):
    print("Stratification maintained for value '0' in validation set")
else:
    print("Stratification not maintained for value '0' in validation set")

if np.round(100 * preprocessed_y[1] / len(df_preprocessed), 1) == np.round(
    100 * val_y[1] / len(val_df), 1
):
    print("Stratification maintained for value '1' in validation set")
else:
    print("Stratification not maintained for value '1' in validation set")

✅ Stratification maintained for value '0' in train set
✅ Stratification maintained for value '1' in train set
✅ Stratification maintained for value '0' in test set
✅ Stratification maintained for value '1' in test set
✅ Stratification maintained for value '0' in validation set
✅ Stratification maintained for value '1' in validation set


Comprobamos la coherencia del número de lesiones

In [ ]:
# numero de lesiones en cada split y porcentaje del original (preprocessed)
print(f"Preprocessed set:\n {df_preprocessed['isic_id'].nunique()} lesions\n")

print(f"Train set:\n {train_df['isic_id'].nunique()} lesions")
print(
    f" {100 * train_df['isic_id'].nunique() / df_preprocessed['isic_id'].nunique()} % \nof preprocessed set\n"
)

print(f"Test set:\n {test_df['isic_id'].nunique()} lesions")
print(
    f" {100 * test_df['isic_id'].nunique() / df_preprocessed['isic_id'].nunique()} % \nof preprocessed set\n"
)

print(f"Validation set:\n {val_df['isic_id'].nunique()} lesions")
print(
    f" {100 * val_df['isic_id'].nunique() / df_preprocessed['isic_id'].nunique()} % \nof preprocessed set\n"
)

# se cumple igualdad de train + test + val = preprocessed
print(
    f"Total lesions in splits: {train_df['isic_id'].nunique() + test_df['isic_id'].nunique() + val_df['isic_id'].nunique()} lesions"
)
print(f"Preprocessed lesions: {df_preprocessed['isic_id'].nunique()} lesions")

if (
    train_df["isic_id"].nunique()
    + test_df["isic_id"].nunique()
    + val_df["isic_id"].nunique()
    == df_preprocessed["isic_id"].nunique()
):
    print("Train + Test + Val = Preprocessed")
else:
    print("Train + Test + Val != Preprocessed")

Preprocessed set:
 381280 lesions

Train set:
 305024 lesions
 80.0 % 
of preprocessed set

Test set:
 38125 lesions
 9.999213176668066 % 
of preprocessed set

Validation set:
 38131 lesions
 10.000786823331934 % 
of preprocessed set

Total lesions in splits: 381280 lesions
Preprocessed lesions: 381280 lesions
✅ Train + Test + Val = Preprocessed


Validamos que hay coherencia con el numero de pacientes

In [ ]:
# numero de pacientes en cada split
print(f"Preprocessed set:\n {df_preprocessed['patient_id'].nunique()} patients\n")

print(f"Train set:\n {train_df['patient_id'].nunique()} patients")
print(
    f" {100 * train_df['patient_id'].nunique() / df_preprocessed['patient_id'].nunique()} % \nof preprocessed set\n"
)

print(f"Test set:\n {test_df['patient_id'].nunique()} patients")
print(
    f" {100 * test_df['patient_id'].nunique() / df_preprocessed['patient_id'].nunique()} % \nof preprocessed set\n"
)

print(f"Validation set:\n {val_df['patient_id'].nunique()} patients")
print(
    f" {100 * val_df['patient_id'].nunique() / df_preprocessed['patient_id'].nunique()} % \nof preprocessed set\n"
)

# Se cumple igualdad de train + test + val = preprocessed
print(
    f"Total patients in splits: {train_df['patient_id'].nunique() + test_df['patient_id'].nunique() + val_df['patient_id'].nunique()} patients"
)
print(f"Preprocessed patients: {df_preprocessed['patient_id'].nunique()} patients")

if (
    train_df["patient_id"].nunique()
    + test_df["patient_id"].nunique()
    + val_df["patient_id"].nunique()
    == df_preprocessed["patient_id"].nunique()
):
    print("Train + Test + Val = Preprocessed")
else:
    print("Train + Test + Val != Preprocessed")

Preprocessed set:
 977 patients

Train set:
 784 patients
 80.24564994882293 % 
of preprocessed set

Test set:
 99 patients
 10.133060388945752 % 
of preprocessed set

Validation set:
 94 patients
 9.62128966223132 % 
of preprocessed set

Total patients in splits: 977 patients
Preprocessed patients: 977 patients
✅ Train + Test + Val = Preprocessed


In [ ]:
# ============================================================
# Hypothesis 2: malignancy classification among biopsied lesions
# ============================================================

# Select only lesions with an available malignancy label
h2_base_df = df_preprocessed.loc[df_preprocessed["target_malignant"].notna()].copy()

h2_train_df = train_df.loc[train_df["target_malignant"].notna()].copy()

h2_test_df = test_df.loc[test_df["target_malignant"].notna()].copy()

h2_val_df = val_df.loc[val_df["target_malignant"].notna()].copy()


# Basic consistency checks

# Hypothesis 2 should only include biopsied lesions
assert h2_base_df["target_biopsy"].eq(1).all(), (
    "H2 contains lesions not labelled as biopsied."
)

# The malignancy target must be binary
assert h2_base_df["target_malignant"].isin([0, 1]).all(), (
    "target_malignant contains values different from 0 and 1."
)

# All H2 lesions must be included exactly once across the three splits
h2_base_ids = set(h2_base_df["isic_id"])
h2_train_ids = set(h2_train_df["isic_id"])
h2_test_ids = set(h2_test_df["isic_id"])
h2_val_ids = set(h2_val_df["isic_id"])

assert not h2_train_ids.intersection(h2_test_ids), (
    "H2 lesion overlap detected between train and test."
)

assert not h2_train_ids.intersection(h2_val_ids), (
    "H2 lesion overlap detected between train and validation."
)

assert not h2_test_ids.intersection(h2_val_ids), (
    "H2 lesion overlap detected between test and validation."
)

assert h2_train_ids.union(h2_test_ids, h2_val_ids) == h2_base_ids, (
    "Some H2 lesions are missing or duplicated across splits."
)


# No patient overlap between splits
h2_train_patients = set(h2_train_df["patient_id"])
h2_test_patients = set(h2_test_df["patient_id"])
h2_val_patients = set(h2_val_df["patient_id"])

assert not h2_train_patients.intersection(h2_test_patients), (
    "H2 patient overlap detected between train and test."
)

assert not h2_train_patients.intersection(h2_val_patients), (
    "H2 patient overlap detected between train and validation."
)

assert not h2_test_patients.intersection(h2_val_patients), (
    "H2 patient overlap detected between test and validation."
)


# Summary of the H2 distribution
def summarize_h2_split(name, df, base_df):
    target_counts = (
        df["target_malignant"].astype(int).value_counts().reindex([0, 1], fill_value=0)
    )

    n_total = len(df)
    n_benign = target_counts[0]
    n_malignant = target_counts[1]

    print(f"\n{name}")
    print("-" * len(name))
    print(f"Lesions: {n_total:,} ({n_total / len(base_df):.2%} of H2)")
    print(f"Patients: {df['patient_id'].nunique():,}")
    print(f"Benign lesions: {n_benign:,} ({n_benign / n_total:.2%})")
    print(f"Malignant lesions: {n_malignant:,} ({n_malignant / n_total:.2%})")
    print(
        "Patients with at least one malignant lesion: "
        f"{df.loc[df['target_malignant'].eq(1), 'patient_id'].nunique():,}"
    )


print("HYPOTHESIS 2 SPLIT VALIDATION")
print("=============================")

print(f"\nComplete H2 subset: {len(h2_base_df):,} lesions")
print(f"Complete H2 subset: {h2_base_df['patient_id'].nunique():,} patients")

summarize_h2_split("Train H2", h2_train_df, h2_base_df)
summarize_h2_split("Validation H2", h2_val_df, h2_base_df)
summarize_h2_split("Test H2", h2_test_df, h2_base_df)

print("\nH2 coverage and patient separation checks passed.")

HYPOTHESIS 2 SPLIT VALIDATION

Complete H2 subset: 1,013 lesions
Complete H2 subset: 629 patients

Train H2
--------
Lesions: 812 (80.16% of H2)
Patients: 497
Benign lesions: 505 (62.19%)
Malignant lesions: 307 (37.81%)
Patients with at least one malignant lesion: 202

Validation H2
-------------
Lesions: 101 (9.97% of H2)
Patients: 68
Benign lesions: 67 (66.34%)
Malignant lesions: 34 (33.66%)
Patients with at least one malignant lesion: 21

Test H2
-------
Lesions: 100 (9.87% of H2)
Patients: 64
Benign lesions: 61 (61.00%)
Malignant lesions: 39 (39.00%)
Patients with at least one malignant lesion: 25

✅ H2 coverage and patient separation checks passed.


In [16]:
from sklearn.model_selection import StratifiedGroupKFold


def split_stratified_group(
    df,
    col_grouping="patient_id",
    col_target="target_biopsy",
    test_val_size=0.2,
    random_state=42,
):
    """
    Split a dataframe while:
    - grouping by patient;
    - stratifying by the selected target;
    - preserving reproducibility.
    """

    n_folds = int(round(1 / test_val_size))

    splitter = StratifiedGroupKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=random_state,
    )

    main_indices, second_indices = next(
        splitter.split(
            X=df,
            y=df[col_target],
            groups=df[col_grouping],
        )
    )

    main_df = df.iloc[main_indices].copy()
    second_df = df.iloc[second_indices].copy()

    return main_df, second_df, main_indices, second_indices

In [17]:
# ============================================================
# Random-state search for the independent H2 split
# H2: benign vs malignant among labelled/biopsied lesions
# ============================================================


def evaluate_h2_random_state(
    df_h2,
    random_state,
    col_grouping="patient_id",
    col_target="target_malignant",
):
    """
    Create an independent 80/10/10 split for H2 and evaluate:
    - split sizes;
    - malignant prevalence;
    - number of patients;
    - absence of patient leakage.
    """

    # First split: 80% train and 20% test-validation
    train_h2, test_val_h2, _, _ = split_stratified_group(
        df_h2,
        col_grouping=col_grouping,
        col_target=col_target,
        test_val_size=0.2,
        random_state=random_state,
    )

    # Second split: divide the remaining 20% into test and validation
    test_h2, val_h2, _, _ = split_stratified_group(
        test_val_h2,
        col_grouping=col_grouping,
        col_target=col_target,
        test_val_size=0.5,
        random_state=random_state,
    )

    splits = {
        "train": train_h2,
        "validation": val_h2,
        "test": test_h2,
    }

    # Check patient leakage
    patient_sets = {name: set(split[col_grouping]) for name, split in splits.items()}

    no_patient_overlap = (
        patient_sets["train"].isdisjoint(patient_sets["validation"])
        and patient_sets["train"].isdisjoint(patient_sets["test"])
        and patient_sets["validation"].isdisjoint(patient_sets["test"])
    )

    base_malignant_rate = df_h2[col_target].mean()

    expected_proportions = {
        "train": 0.80,
        "validation": 0.10,
        "test": 0.10,
    }

    result = {
        "random_state": random_state,
        "base_malignant_pct": 100 * base_malignant_rate,
        "no_patient_overlap": no_patient_overlap,
    }

    prevalence_error = 0
    size_error = 0

    for split_name, split_df in splits.items():
        n_lesions = len(split_df)
        n_patients = split_df[col_grouping].nunique()

        counts = (
            split_df[col_target]
            .astype(int)
            .value_counts()
            .reindex([0, 1], fill_value=0)
        )

        n_benign = int(counts[0])
        n_malignant = int(counts[1])
        malignant_rate = n_malignant / n_lesions
        split_proportion = n_lesions / len(df_h2)

        prevalence_error += abs(malignant_rate - base_malignant_rate)

        size_error += abs(split_proportion - expected_proportions[split_name])

        result[f"{split_name}_n"] = n_lesions
        result[f"{split_name}_patients"] = n_patients
        result[f"{split_name}_benign_n"] = n_benign
        result[f"{split_name}_malignant_n"] = n_malignant
        result[f"{split_name}_malignant_pct"] = 100 * malignant_rate
        result[f"{split_name}_size_pct"] = 100 * split_proportion

    result["prevalence_error"] = 100 * prevalence_error
    result["size_error"] = 100 * size_error

    # Prevalence is prioritised over small deviations from 80/10/10
    result["score"] = result["prevalence_error"] + 0.5 * result["size_error"]

    return result


# H2 contains only lesions with a known malignant label
df_h2 = df_preprocessed.loc[df_preprocessed["target_malignant"].notna()].copy()

assert df_h2["target_malignant"].isin([0, 1]).all()
assert df_h2["target_biopsy"].eq(1).all()

# Search across a moderate range of seeds
h2_seed_results = pd.DataFrame(
    [
        evaluate_h2_random_state(
            df_h2=df_h2,
            random_state=seed,
        )
        for seed in range(100)
    ]
)

h2_seed_results = (
    h2_seed_results.query("no_patient_overlap")
    .sort_values(
        ["score", "prevalence_error", "size_error"],
        ascending=True,
    )
    .reset_index(drop=True)
)

columns_to_show = [
    "random_state",
    "base_malignant_pct",
    "train_n",
    "train_malignant_pct",
    "validation_n",
    "validation_malignant_pct",
    "test_n",
    "test_malignant_pct",
    "prevalence_error",
    "size_error",
    "score",
]

h2_seed_results[columns_to_show].head(15)

,random_state,base_malignant_pct,train_n,train_malignant_pct,validation_n,validation_malignant_pct,test_n,test_malignant_pct,prevalence_error,size_error,score
0,37,37.51234,811,37.484587,101,37.623762,101,37.623762,0.250598,0.118460,0.309828
1,44,37.51234,811,37.484587,101,37.623762,101,37.623762,0.250598,0.118460,0.309828
2,81,37.51234,811,37.484587,101,37.623762,101,37.623762,0.250598,0.118460,0.309828
3,99,37.51234,811,37.484587,101,37.623762,101,37.623762,0.250598,0.118460,0.309828
4,10,37.51234,810,37.530864,102,37.254902,101,37.623762,0.387385,0.138203,0.456487
5,27,37.51234,810,37.530864,102,37.254902,101,37.623762,0.387385,0.138203,0.456487
6,33,37.51234,810,37.530864,102,37.254902,101,37.623762,0.387385,0.138203,0.456487
7,61,37.51234,810,37.530864,102,37.254902,101,37.623762,0.387385,0.138203,0.456487
8,70,37.51234,810,37.530864,101,37.623762,102,37.254902,0.387385,0.138203,0.456487
9,73,37.51234,810,37.530864,101,37.623762,102,37.254902,0.387385,0.138203,0.456487


In [ ]:
def validate_h2_split(
    df_base,
    train_df,
    val_df,
    test_df,
    group_col="patient_id",
    target_col="target_malignant",
):
    splits = {
        "train": train_df,
        "validation": val_df,
        "test": test_df,
    }

    # Patient leakage
    patient_sets = {name: set(df[group_col]) for name, df in splits.items()}

    assert patient_sets["train"].isdisjoint(patient_sets["validation"])
    assert patient_sets["train"].isdisjoint(patient_sets["test"])
    assert patient_sets["validation"].isdisjoint(patient_sets["test"])

    # Lesion coverage
    base_ids = set(df_base["isic_id"])
    split_ids = {name: set(df["isic_id"]) for name, df in splits.items()}

    assert split_ids["train"].isdisjoint(split_ids["validation"])
    assert split_ids["train"].isdisjoint(split_ids["test"])
    assert split_ids["validation"].isdisjoint(split_ids["test"])
    assert set().union(*split_ids.values()) == base_ids

    # Summary
    base_positive_rate = df_base[target_col].mean()

    print("HYPOTHESIS 2 SPLIT VALIDATION")
    print("=============================")
    print(
        f"Base: {len(df_base):,} lesions | "
        f"{df_base[group_col].nunique():,} patients | "
        f"malignant: {base_positive_rate:.2%}"
    )

    for name, df in splits.items():
        counts = df[target_col].astype(int).value_counts().reindex([0, 1], fill_value=0)

        print(f"\n{name.capitalize()}")
        print("-" * len(name))
        print(f"Lesions: {len(df):,} ({len(df) / len(df_base):.2%})")
        print(f"Patients: {df[group_col].nunique():,}")
        print(f"Benign: {counts[0]:,} ({counts[0] / len(df):.2%})")
        print(f"Malignant: {counts[1]:,} ({counts[1] / len(df):.2%})")

    print("\nH2 split validation passed.")

In [23]:
best_h2_random_state = int(h2_seed_results.loc[0, "random_state"])

train_h2_df, test_val_h2_df, _, _ = split_stratified_group(
    df_h2,
    col_grouping="patient_id",
    col_target="target_malignant",
    test_val_size=0.2,
    random_state=best_h2_random_state,
)

test_h2_df, val_h2_df, _, _ = split_stratified_group(
    test_val_h2_df,
    col_grouping="patient_id",
    col_target="target_malignant",
    test_val_size=0.5,
    random_state=best_h2_random_state,
)

validate_h2_split(
    df_base=df_h2,
    train_df=train_h2_df,
    val_df=val_h2_df,
    test_df=test_h2_df,
)

HYPOTHESIS 2 SPLIT VALIDATION
Base: 1,013 lesions | 629 patients | malignant: 37.51%

Train
-----
Lesions: 811 (80.06%)
Patients: 504
Benign: 507 (62.52%)
Malignant: 304 (37.48%)

Validation
----------
Lesions: 101 (9.97%)
Patients: 64
Benign: 63 (62.38%)
Malignant: 38 (37.62%)

Test
----
Lesions: 101 (9.97%)
Patients: 61
Benign: 63 (62.38%)
Malignant: 38 (37.62%)

✅ H2 split validation passed.


## Conclusiones sobre la partición de los datos

Se definen particiones independientes para las dos hipótesis del proyecto, ya que cada una utiliza una variable objetivo y una población de análisis diferentes.

Para la **hipótesis 1**, orientada a predecir si una lesión debe ser biopsiada, se utiliza el conjunto completo de 381.280 lesiones y se estratifica por `target_biopsy`, agrupando por `patient_id`. Con `random_state=42` se obtiene una distribución prácticamente exacta del 80 % para entrenamiento, 10 % para validación y 10 % para test. Además, se mantiene la proporción de la clase positiva y no existe solapamiento de pacientes entre particiones.

Para la **hipótesis 2**, orientada a clasificar las lesiones biopsiadas como benignas o malignas, se consideran únicamente las 1.013 lesiones con un valor disponible en `target_malignant`. Debido a que se trata de un subconjunto diferente, se realiza una partición independiente, estratificada directamente por esta variable y agrupada igualmente por paciente. Tras evaluar diferentes semillas, `random_state=37` proporciona una distribución cercana al 80/10/10 y conserva prácticamente sin variaciones la proporción global de lesiones malignas en los tres conjuntos.

En ambas hipótesis, todas las lesiones quedan asignadas a una única partición y no existe solapamiento de pacientes entre entrenamiento, validación y test. Por tanto, las particiones obtenidas se consideran adecuadas para el posterior entrenamiento y evaluación de los modelos, evitando el riesgo de *data leakage* entre conjuntos.
